# IMC Prosperity 4 — Round Data Analysis
**Goal**: Analyze market dynamics BEFORE writing strategy code. Find hidden edges.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from collections import Counter, defaultdict

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)

# === CONFIGURE THESE ===
ROUND = 0
DAY = -1
DATA_DIR = 'imc-prosperity-4-backtester-kevin/prosperity4bt/resources'

# Load price data
prices_df = pd.read_csv(f'{DATA_DIR}/round{ROUND}/prices_round_{ROUND}_day_{DAY}.csv', sep=';')
print(f'Products: {prices_df["product"].unique()}')
print(f'Timestamps: {prices_df["timestamp"].nunique()}')
prices_df.head()

In [ ]:
# Load trade data
trades_df = pd.read_csv(f'{DATA_DIR}/round{ROUND}/trades_round_{ROUND}_day_{DAY}.csv', sep=';')
trades_df.columns = ['timestamp', 'buyer', 'seller', 'symbol', 'currency', 'price', 'quantity']
print(f'Total trades: {len(trades_df)}')
print(f'Trades per product:')
print(trades_df['symbol'].value_counts())
trades_df.head()

## 1. Price Series Visualization

In [ ]:
# Plot mid price for each product
products = prices_df['product'].unique()
fig, axes = plt.subplots(len(products), 1, figsize=(14, 4*len(products)))
if len(products) == 1:
    axes = [axes]

for ax, product in zip(axes, products):
    prod_data = prices_df[prices_df['product'] == product]
    ax.plot(prod_data['timestamp'], prod_data['mid_price'], linewidth=0.5)
    ax.set_title(f'{product} — Mid Price')
    ax.set_xlabel('Timestamp')
    ax.set_ylabel('Price')

plt.tight_layout()
plt.show()

## 2. Spread Analysis

In [ ]:
# Compute spread (best ask - best bid) per product
for product in products:
    prod = prices_df[prices_df['product'] == product].copy()
    prod['spread'] = prod['ask_price_1'] - prod['bid_price_1']
    print(f'\n{product} spread stats:')
    print(prod['spread'].describe())
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3))
    ax1.plot(prod['timestamp'], prod['spread'], linewidth=0.5)
    ax1.set_title(f'{product} — Spread over time')
    ax2.hist(prod['spread'], bins=30, edgecolor='black')
    ax2.set_title(f'{product} — Spread distribution')
    plt.tight_layout()
    plt.show()

## 3. Returns & Autocorrelation

In [ ]:
# Compute returns and test autocorrelation
for product in products:
    prod = prices_df[prices_df['product'] == product].copy()
    mid = prod['mid_price'].values
    returns = np.diff(mid)
    
    print(f'\n=== {product} ===')
    print(f'Return stats: mean={returns.mean():.4f}, std={returns.std():.4f}')
    
    # Autocorrelation at lags 1-20
    lags = range(1, 21)
    autocorrs = [np.corrcoef(returns[:-lag], returns[lag:])[0,1] for lag in lags]
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 3))
    ax1.hist(returns, bins=50, edgecolor='black')
    ax1.set_title(f'{product} — Return distribution')
    
    ax2.bar(lags, autocorrs)
    ax2.axhline(y=0, color='black', linewidth=0.5)
    ax2.axhline(y=2/np.sqrt(len(returns)), color='red', linestyle='--', linewidth=0.5)
    ax2.axhline(y=-2/np.sqrt(len(returns)), color='red', linestyle='--', linewidth=0.5)
    ax2.set_title(f'{product} — Autocorrelation (red=95% CI)')
    ax2.set_xlabel('Lag')
    
    # Cumulative returns
    ax3.plot(np.cumsum(returns), linewidth=0.5)
    ax3.set_title(f'{product} — Cumulative returns')
    
    plt.tight_layout()
    plt.show()

## 4. Trade Size Analysis (Find Olivia-like patterns)

In [ ]:
# Trade size histogram per product
for product in products:
    prod_trades = trades_df[trades_df['symbol'] == product]
    if len(prod_trades) == 0:
        print(f'{product}: no trades')
        continue
    
    print(f'\n=== {product} — {len(prod_trades)} trades ===')
    print('Size distribution:')
    print(prod_trades['quantity'].value_counts().head(10))
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 3))
    ax.hist(prod_trades['quantity'], bins=range(1, prod_trades['quantity'].max()+2), edgecolor='black')
    ax.set_title(f'{product} — Trade size distribution')
    ax.set_xlabel('Quantity')
    plt.tight_layout()
    plt.show()

## 5. Trader Profitability Analysis

In [ ]:
# For each trader (buyer/seller), compute their win rate
# A 'win' = bought below future mid, or sold above future mid
# Use mid price 5 ticks ahead as 'future'

for product in products:
    prod_trades = trades_df[trades_df['symbol'] == product].copy()
    prod_prices = prices_df[prices_df['product'] == product][['timestamp', 'mid_price']]
    
    if len(prod_trades) == 0:
        continue
    
    # Merge to get mid at trade time
    prod_trades = prod_trades.merge(prod_prices, on='timestamp', how='left')
    
    # Get future mid (5 ticks = 500 timestamps ahead)
    future_map = dict(zip(prod_prices['timestamp'], prod_prices['mid_price'].shift(-5)))
    prod_trades['future_mid'] = prod_trades['timestamp'].map(future_map)
    prod_trades = prod_trades.dropna(subset=['future_mid'])
    
    # Buyer analysis
    buyers = prod_trades[prod_trades['buyer'] != ''].copy()
    if len(buyers) > 0:
        buyers['won'] = buyers['future_mid'] > buyers['price']
        buyer_stats = buyers.groupby('buyer').agg(
            trades=('won', 'count'),
            win_rate=('won', 'mean'),
            avg_qty=('quantity', 'mean')
        ).sort_values('win_rate', ascending=False)
        print(f'\n=== {product} — Buyer win rates ===')
        print(buyer_stats[buyer_stats['trades'] >= 5])
    
    # Seller analysis
    sellers = prod_trades[prod_trades['seller'] != ''].copy()
    if len(sellers) > 0:
        sellers['won'] = sellers['future_mid'] < sellers['price']
        seller_stats = sellers.groupby('seller').agg(
            trades=('won', 'count'),
            win_rate=('won', 'mean'),
            avg_qty=('quantity', 'mean')
        ).sort_values('win_rate', ascending=False)
        print(f'\n=== {product} — Seller win rates ===')
        print(seller_stats[seller_stats['trades'] >= 5])

## 6. Trades at Extremes (Frankfurt's Olivia Detection)

In [ ]:
# Track running daily min/max and flag trades occurring at extremes
for product in products:
    prod = prices_df[prices_df['product'] == product].copy()
    prod_trades = trades_df[trades_df['symbol'] == product].copy()
    
    if len(prod_trades) == 0:
        continue
    
    mid_series = prod.set_index('timestamp')['mid_price']
    
    # Compute running min/max
    running_min = mid_series.expanding().min()
    running_max = mid_series.expanding().max()
    
    # For each trade, check if price is at/near running extreme
    extreme_trades = []
    for _, trade in prod_trades.iterrows():
        ts = trade['timestamp']
        if ts not in running_min.index:
            continue
        rmin = running_min[ts]
        rmax = running_max[ts]
        price = trade['price']
        qty = trade['quantity']
        
        at_low = abs(price - rmin) <= 2
        at_high = abs(price - rmax) <= 2
        
        if at_low or at_high:
            extreme_trades.append({
                'timestamp': ts,
                'price': price,
                'quantity': qty,
                'buyer': trade['buyer'],
                'seller': trade['seller'],
                'at_low': at_low,
                'at_high': at_high,
                'running_min': rmin,
                'running_max': rmax
            })
    
    ext_df = pd.DataFrame(extreme_trades)
    if len(ext_df) > 0:
        print(f'\n=== {product} — Trades at extremes: {len(ext_df)} ===')
        print(f'At low: {ext_df["at_low"].sum()}, At high: {ext_df["at_high"].sum()}')
        print(f'\nQuantity distribution at extremes:')
        print(ext_df['quantity'].value_counts().head(10))
        print(f'\nSample extreme trades:')
        print(ext_df.head(10))
    else:
        print(f'{product}: no trades at extremes')

## 7. Order Book Imbalance

In [ ]:
# Does bid/ask volume imbalance predict next price move?
for product in products:
    prod = prices_df[prices_df['product'] == product].copy()
    prod['bid_vol'] = prod['bid_volume_1'].fillna(0) + prod['bid_volume_2'].fillna(0) + prod['bid_volume_3'].fillna(0)
    prod['ask_vol'] = prod['ask_volume_1'].fillna(0) + prod['ask_volume_2'].fillna(0) + prod['ask_volume_3'].fillna(0)
    prod['imbalance'] = (prod['bid_vol'] - prod['ask_vol']) / (prod['bid_vol'] + prod['ask_vol'])
    prod['future_return'] = prod['mid_price'].shift(-1) - prod['mid_price']
    
    corr = prod['imbalance'].corr(prod['future_return'])
    print(f'{product}: imbalance-return correlation = {corr:.4f}')
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.scatter(prod['imbalance'], prod['future_return'], alpha=0.1, s=1)
    ax.set_xlabel('Book Imbalance')
    ax.set_ylabel('Next-tick Return')
    ax.set_title(f'{product} — Imbalance vs Future Return (corr={corr:.4f})')
    plt.tight_layout()
    plt.show()

## 8. Volatility Regimes

In [ ]:
# Rolling volatility — identify regime changes
WINDOW = 50

for product in products:
    prod = prices_df[prices_df['product'] == product].copy()
    returns = prod['mid_price'].diff()
    rolling_vol = returns.rolling(WINDOW).std()
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
    ax1.plot(prod['timestamp'].values, prod['mid_price'].values, linewidth=0.5)
    ax1.set_title(f'{product} — Price')
    ax2.plot(prod['timestamp'].values, rolling_vol.values, linewidth=0.5, color='orange')
    ax2.set_title(f'{product} — Rolling {WINDOW}-tick Volatility')
    plt.tight_layout()
    plt.show()

## 9. Cross-Product Correlation

In [ ]:
# Do products move together? Does one lead the other?
if len(products) >= 2:
    pivot = prices_df.pivot_table(index='timestamp', columns='product', values='mid_price')
    returns_pivot = pivot.diff()
    
    print('Contemporaneous correlation:')
    print(returns_pivot.corr())
    
    # Lead-lag: does product A's return predict product B's next return?
    print('\nLead-lag correlations (row leads column by 1 tick):')
    for p1 in products:
        for p2 in products:
            if p1 != p2:
                corr = returns_pivot[p1].corr(returns_pivot[p2].shift(-1))
                if abs(corr) > 0.02:
                    print(f'  {p1} → {p2} (1-tick lag): {corr:.4f}')

## 10. Trade Timing Patterns

In [ ]:
# Are there periodic patterns in when trades occur?
for product in products:
    prod_trades = trades_df[trades_df['symbol'] == product]
    if len(prod_trades) == 0:
        continue
    
    # Inter-trade time distribution
    inter_trade = prod_trades['timestamp'].diff().dropna()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3))
    ax1.hist(inter_trade, bins=50, edgecolor='black')
    ax1.set_title(f'{product} — Inter-trade time distribution')
    ax1.set_xlabel('Ticks between trades')
    
    # Trade count per 1000-tick bucket (intraday seasonality)
    buckets = (prod_trades['timestamp'] // 1000) * 1000
    bucket_counts = buckets.value_counts().sort_index()
    ax2.bar(bucket_counts.index, bucket_counts.values, width=800)
    ax2.set_title(f'{product} — Trades per 1000-tick window')
    ax2.set_xlabel('Timestamp bucket')
    
    plt.tight_layout()
    plt.show()